### Comp_outcome

In [1]:
%load_ext autoreload
%autoreload 2
## Imports

import sys
sys.path.append("..") # Adds the project root to the path

In [2]:
from src.data_exposure import get_exposure
from src.data_hazard import get_haz_dict
from src.data_insurance import get_insurance
from src.helpers import comp_impact, comp_damage_map, comp_who_pays


haz_dict = get_haz_dict()
haz_list = list(haz_dict.keys())

exposure = get_exposure(hazard_types=haz_list)

from src.helpers import comp_impact

df = exposure.gdf
df["eai"] = comp_impact(haz_dict=haz_dict,
                        exposure=exposure)

df["insurance_level"] = get_insurance(0.3)

df["damaged_area"] = comp_damage_map(
    eai=df["eai"],
    area=df["area"],
    value=df["value"]
)

df =df.join(
    comp_who_pays(
        relative_damage=df["eai"],
        insured=df["insurance_level"]
    )
)

df

KeyboardInterrupt: 

In [ ]:
import numpy as np

who_pays = df[["F", "I", "G"]]
damaged_area = df["damaged_area"]

result = df[["F", "I", "G"]].multiply(df["damaged_area"], axis=0)

result.sum() / damaged_area.sum()

F    0.455168
I    0.030186
G    0.514646
dtype: float64

In [ ]:
import numpy as np

def comp_outcome (damaged_area, who_pays):

    ## Which Actor is responsible for how much area in each Departement
    result = who_pays.multiply(damaged_area, axis=0)

    ## Aggregate over all of France
    absolute = result.sum()
    relative = absolute / damaged_area.sum()
    
    return {
        "who_pays_area": result,
        "agg_absolute": absolute,
        "agg_relative": relative
    }

In [ ]:
outcome = comp_outcome(
    damaged_area=df["damaged_area"], 
    who_pays=df[["F", "I", "G"]]
)

In [ ]:
outcome["agg_absolute"]

F    54192.535669
I     3593.911191
G    61273.954773
dtype: float64